# 2. Getting your text ready

In the [previous notebook](01-what-is-a-formula.ipynb) we looked at *what* a formulaic phrase is. Before any software can help you find formulas in your own material, your text needs to be turned into a shape the software can work with. This notebook walks through that step, called **tokenization**, and through loading a real dataset of historical documents into the format this toolkit expects.

No prior programming knowledge is assumed &mdash; we'll explain each piece of code as we go.

## Why text needs to be split into words

To you, a sentence is obviously made up of words. To a computer, a sentence is just a long string of characters &mdash; it has no built-in idea of where one word ends and the next begins. **Tokenization** is the process of splitting that string of characters into pieces (usually words), called **tokens**, that the software can then count, compare, and search through.

The simplest possible way to do this is to split on spaces. Let's try it on one of the resolution openings from the first notebook.

In [1]:
text = "ONtfangen een Missive van den Heere Gallieris, haar Hoog Mogende Minister."

tokens = text.split()
print(tokens)

['ONtfangen', 'een', 'Missive', 'van', 'den', 'Heere', 'Gallieris,', 'haar', 'Hoog', 'Mogende', 'Minister.']


That mostly worked &mdash; but look closely at two of the tokens: `'Gallieris,'` and `'Minister.'` still have punctuation stuck to them. To a human reader that's invisible; to the software, `'Minister.'` (with a full stop) and `'Minister'` (without one, perhaps from a different sentence) are two completely different tokens, even though they're obviously the same word. If we don't fix this, the software will undercount how often "Minister" really occurs, and will fail to recognise a formula whenever it happens to end a sentence.

There's a second, similar problem: capitalisation. The word at the very start of this formula is written `ONtfangen` here (a common way of marking the start of a new entry in this kind of document) but appears as plain `ontfangen` everywhere else in the same corpus. If we treat `ONtfangen` and `ontfangen` as different words, we'll miss most of the formula's occurrences.

## A better way to split text into tokens

We can fix both problems at once: lowercase everything first, and split on *letters* instead of on spaces &mdash; that is, treat any run of letters as one token, and throw away anything that isn't a letter (commas, full stops, numbers, etc.).

The code below does this using a small pattern-matching tool built into Python (the `re`, "regular expressions", module). You don't need to understand the pattern itself in detail &mdash; just that `[A-Za-z\u00c0-\u00ff]+` means "one or more letters, including accented letters like é, ë, ÿ".

In [2]:
import re

TOKEN_PATTERN = re.compile(r"[A-Za-z\u00c0-\u00ff]+")


def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())


tokens = tokenize(text)
print(tokens)

['ontfangen', 'een', 'missive', 'van', 'den', 'heere', 'gallieris', 'haar', 'hoog', 'mogende', 'minister']


Now every token is a clean, lowercase word with no punctuation attached. This `tokenize` function is simple, but it's exactly the kind of function you'll use throughout this series &mdash; and you're welcome to adjust the pattern if your own material has different quirks (e.g. different accented characters, or hyphens you want to keep).

## An alternative: keeping punctuation as its own tokens

The tokenizer above throws punctuation away entirely. That's a reasonable default, but it's worth pausing on: **punctuation can itself be a meaningful signal**, not just clutter.

Look again at the very first paragraph in our dataset, an attendance list:

> Luna den 4 January 1768. PRAESIDE, Den Heere van Palland tot Olinthuys. PRAESENTIBUS, De Heeren van Lynden tot Hemmen,

The full stops here mark the boundary between "today's date", "who presided", and "who else was present" &mdash; three distinct pieces of information glued together in one paragraph. The commas mark where one item in a list of names ends and the next begins. If we discard all punctuation, that structure disappears completely, and we can never get it back later. This matters in particular for [notebook 5](05-where-do-documents-begin-and-end.ipynb), where we'll use exactly this kind of structural signal to help find where one formula or document ends and the next begins.

So instead of matching only letters, we can match *either* a run of letters, *or* a run of digits, *or* a single punctuation character &mdash; keeping every punctuation mark as its own separate token, rather than throwing it away.

In [3]:
PUNCT_TOKEN_PATTERN = re.compile(r"[A-Za-z\u00c0-\u00ff]+|\d+|[^\sA-Za-z\u00c0-\u00ff\d]")


def tokenize_keep_punctuation(text):
    return PUNCT_TOKEN_PATTERN.findall(text.lower())


attendance_text = (
    "Luna den 4 January 1768. PRAESIDE, Den Heere van Palland tot Olinthuys. "
    "PRAESENTIBUS, De Heeren van Lynden tot Hemmen,"
)

print('Punctuation discarded:')
print(tokenize(attendance_text))
print()
print('Punctuation kept as separate tokens:')
print(tokenize_keep_punctuation(attendance_text))

Punctuation discarded:
['luna', 'den', 'january', 'praeside', 'den', 'heere', 'van', 'palland', 'tot', 'olinthuys', 'praesentibus', 'de', 'heeren', 'van', 'lynden', 'tot', 'hemmen']

Punctuation kept as separate tokens:
['luna', 'den', '4', 'january', '1768', '.', 'praeside', ',', 'den', 'heere', 'van', 'palland', 'tot', 'olinthuys', '.', 'praesentibus', ',', 'de', 'heeren', 'van', 'lynden', 'tot', 'hemmen', ',']


Notice that `.` and `,` now show up as their own tokens, sitting between the words they separated, rather than disappearing. The word tokens themselves are identical either way &mdash; we haven't lost anything, we've only *kept* something extra that the first approach threw away.

**The trade-off:** keeping punctuation gives the software a few more distinct token types to deal with (each punctuation mark becomes its own "word"), and any later step that counts or compares phrases needs to decide whether `"olinthuys"` followed by `"."` should count as part of the same phrase as `"olinthuys"` on its own. There's no universally correct answer here &mdash; it depends on your research question. If you're mainly interested in the *wording* of formulas, discarding punctuation (as the rest of this series does, for simplicity) is usually fine. If punctuation itself is part of what you're studying &mdash; for example, because list-like structures (commas), abbreviations, or sentence-final punctuation help you tell where a formula begins or ends &mdash; keeping it lets you use that signal later, at the cost of a slightly larger, slightly noisier vocabulary.

Everything in this toolkit works with whichever tokenization you choose; it has no built-in preference. The rest of this series uses the punctuation-discarding `tokenize` function for simplicity, but if your own research question is sensitive to punctuation, `tokenize_keep_punctuation` (or your own variation on it) is a reasonable place to start instead.

One more thing worth noticing in the comparison above: the original `tokenize` function also silently drops *numbers* (the day, `4`, and the year, `1768`, are both missing from its output), because its pattern only matches letters. For a historian, dates and other numbers are often exactly the kind of detail you don't want to lose — `tokenize_keep_punctuation` keeps them too, as its own tokens, for the same reason it keeps punctuation.

## From one sentence to a real dataset

Real historical research rarely involves just one sentence — usually you have a whole collection of documents, often supplied as a spreadsheet-like file with one row per document or paragraph. The example dataset used throughout this series is exactly that: a gzip-compressed tab-separated file (`.tsv.gz`) with one row per paragraph of a set of 1768 resolutions, and columns for the session date, the resolution it belongs to, the paragraph's position in that resolution, its location on the scanned page, and the transcribed text itself. (This notebook assumes Jupyter was started from the repository root, so the data file can be found at the relative path `data/resolution_paragraphs/...`; adjust the path below if you launched Jupyter from elsewhere.)

Let's load a few rows and look at what's actually in there.

In [4]:
import csv
import gzip

rows = []
with gzip.open('data/resolution_paragraphs/paragraphs-3823.tsv.gz', 'rt', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter='\t'):
        rows.append(row)

print(f'{len(rows)} rows loaded')
print()
print('Columns:', list(rows[0].keys()))
print()
print('First row:')
for key, value in rows[0].items():
    print(f'  {key}: {value}')

2909 rows loaded

Columns: ['session_date', 'resolution_id', 'para_id', 'line_start', 'line_end', 'text']

First row:
  session_date: 1768-01-04
  resolution_id: session-3823-num-3-attendance_list
  para_id: session-3823-num-3-para-1
  line_start: NL-HaNA_1.01.02_3823_0086-line-2617-1931-570-57
  line_end: NL-HaNA_1.01.02_3823_0086-line-2505-2200-852-54
  text: Luna den 4 January 1768. PRAESIDE, Den Heere van Palland tot Olinthuys. PRAESENTIBUS, De Heeren van Lynden tot Hemmen,


Each row is a Python `dict` (short for dictionary) &mdash; a way of storing labelled values, here labelled by column name, so `row['text']` gives you the transcribed text of that row, `row['session_date']` gives you its date, and so on. You don't need to understand dictionaries in depth; just remember that `row['text']` is how we'll always get at the actual words.

Now let's tokenize the text of every row, using the `tokenize` function we wrote above.

In [5]:
documents = [tokenize(row['text']) for row in rows]

print(f'{len(documents)} documents (one per paragraph)')
print()
print('First document, as a list of tokens:')
print(documents[0])

2909 documents (one per paragraph)

First document, as a list of tokens:
['luna', 'den', 'january', 'praeside', 'den', 'heere', 'van', 'palland', 'tot', 'olinthuys', 'praesentibus', 'de', 'heeren', 'van', 'lynden', 'tot', 'hemmen']


## What counts as a "document"?

We've been calling each tokenized paragraph a **document**. That word is used loosely throughout this toolkit &mdash; a "document" can be a paragraph, a whole resolution, a sentence, a page, or any other unit you choose to treat as one piece of text. What matters is this: **the software treats different documents as separate**, and will not assume that the end of one document continues into the start of the next. If you're searching for repeated phrases *within* one paragraph, you'd treat each paragraph as a document, as we just did. If you instead want to look *across* paragraph boundaries (for example, because a sentence sometimes continues into the next paragraph in the original source), you'd choose a different, larger unit.

Later notebooks in this series (especially [notebook 5](05-where-do-documents-begin-and-end.ipynb), on finding document boundaries) revisit this choice in more depth, because sometimes you genuinely don't know in advance where one historical document ends and the next begins &mdash; finding that out is itself one of the things this toolkit can help with.

## A quick sanity check

Before moving on, it's worth checking that tokenization actually solved the problem it set out to solve: does our formula opener now look the same regardless of capitalisation, across many different paragraphs?

In [6]:
starts_with_opener = sum(
    1 for doc in documents
    if doc[:3] == ['ontfangen', 'een', 'missive']
)
print(f'{starts_with_opener} of {len(documents)} paragraphs start with "ontfangen een missive"')

1263 of 2909 paragraphs start with "ontfangen een missive"


Several hundred paragraphs in this single file alone open with this exact sequence of three tokens &mdash; and because we lowercased everything, this count includes paragraphs that originally had it written as `Ontfangen`, `ONtfangen`, or any other capitalisation. That's the payoff of getting tokenization right before doing anything else.

## Next steps

You now have a list of tokenized documents &mdash; exactly the input the toolkit's phrase-search tool expects. In [notebook 3](03-finding-repeated-phrases.ipynb), we'll use it to let the software discover candidate formulaic phrases automatically, rather than relying on us already knowing what to look for.